### 1.


In [1]:
def substitute(expr, var, value):
    if isinstance(expr, tuple):
        return tuple(substitute(x, var, value) for x in expr)
    if isinstance(expr, list):
        return [substitute(x, var, value) for x in expr]
    if expr == var:
        return value
    return expr


def remove_implication(expr):
    op = expr[0]

    if op == 'implies':
        return ('or',
                ('not', remove_implication(expr[1])),
                remove_implication(expr[2]))

    if op in ('and', 'or'):
        return (op, remove_implication(expr[1]),
                remove_implication(expr[2]))

    if op in ('forall', 'exists'):
        return (op, expr[1], remove_implication(expr[2]))

    return expr


def move_negations(expr, neg=False):
    op = expr[0]

    if op == 'not':
        return move_negations(expr[1], not neg)

    if op in ('forall', 'exists'):
        new_op = op
        if neg:
            new_op = 'exists' if op == 'forall' else 'forall'
        return (new_op, expr[1], move_negations(expr[2], neg))

    if op in ('and', 'or'):
        new_op = op
        if neg:
            new_op = 'or' if op == 'and' else 'and'

        return (new_op,
                move_negations(expr[1], neg),
                move_negations(expr[2], neg))

    if neg:
        return ('not', expr)

    return expr


def skolemize(expr, universal=[]):
    op = expr[0]

    if op == 'forall':
        return skolemize(expr[2], universal + [expr[1]])

    if op == 'exists':
        var = expr[1]

        if universal:
            value = ('func', 'f', universal)
        else:
            value = ('const', 'A')

        body = substitute(expr[2], var, value)
        return skolemize(body, universal)

    if op in ('and', 'or'):
        return (op,
                skolemize(expr[1], universal),
                skolemize(expr[2], universal))

    return expr


def distribute(expr):
    if expr[0] == 'and':
        return ('and', distribute(expr[1]), distribute(expr[2]))

    if expr[0] == 'or':
        left = distribute(expr[1])
        right = distribute(expr[2])

        if left[0] == 'and':
            return (
                'and',
                distribute(('or', left[1], right)),
                distribute(('or', left[2], right))
            )

        if right[0] == 'and':
            return (
                'and',
                distribute(('or', left, right[1])),
                distribute(('or', left, right[2]))
            )

        return ('or', left, right)

    return expr


def display(expr):
    if not isinstance(expr, tuple):
        return str(expr)

    op = expr[0]

    if op == 'pred':
        return expr[1] + "(" + ", ".join(map(display, expr[2])) + ")"

    if op == 'func':
        return expr[1] + "(" + ", ".join(expr[2]) + ")"

    if op == 'const':
        return expr[1]

    if op == 'not':
        return "NOT " + display(expr[1])

    if op in ('and', 'or'):
        symbol = ' AND ' if op == 'and' else ' OR '
        return "(" + display(expr[1]) + symbol + display(expr[2]) + ")"

    return display(expr[2])


# Original FOPL expression
P = ('pred', 'P', ['x'])
Q = ('pred', 'Q', ['y'])
R = ('pred', 'R', ['x', 'y'])

expression = (
    'forall', 'x',
    ('implies', P, ('exists', 'y', ('and', Q, R)))
)

print("Original Expression:")
print("FORALL x: P(x) -> EXISTS y: (Q(y) AND R(x,y))")

# Step 1: Remove implication
step1 = remove_implication(expression)
print("\nAfter removing implication:")
print(display(step1))

# Step 2: Move negations inward
step2 = move_negations(step1)
print("\nAfter moving negations:")
print(display(step2))

# Step 3: Skolemization
step3 = skolemize(step2)
print("\nAfter Skolemization:")
print(display(step3))

# Step 4: Convert to CNF
step4 = distribute(step3)
print("\nFinal CNF:")
print(display(step4))

Original Expression:
FORALL x: P(x) -> EXISTS y: (Q(y) AND R(x,y))

After removing implication:
(NOT P(x) OR (Q(y) AND R(x, y)))

After moving negations:
(NOT P(x) OR (Q(y) AND R(x, y)))

After Skolemization:
(NOT P(x) OR (Q(f(x)) AND R(x, f(x))))

Final CNF:
((NOT P(x) OR Q(f(x))) AND (NOT P(x) OR R(x, f(x))))
